In [2]:
## PART 1::
class Customer:
    def __init__(self, id: int, revenue: int):
        self.id = id
        self.revenue = revenue
        self.total_revenue = revenue
        self.referrals = []

class RevenueSystem:
    def __init__(self):
        self.customers = {}
        self.next_id = 0

    def insert(self, revenue: int, referrer_id: int = None) -> int:
        """
        Add a new customer.
        Time Complexity: O(1)
        """
        customer_id = self.next_id
        self.next_id += 1

        customer = Customer(customer_id, revenue)
        self.customers[customer_id] = customer

        if referrer_id is not None:
            # Update the person who referred them
            referrer = self.customers[referrer_id]
            referrer.total_revenue += revenue
            referrer.referrals.append(customer_id)

        return customer_id

    def get_lowest_k_by_total_revenue(self, k: int, min_total_revenue: int) -> set:
        """
        Sort and find the top K customers on demand.
        Time Complexity: O(n log n)
        Space Complexity: O(n)
        """
        # 1. Find everyone who meets the minimum money requirement
        eligible = [
            customer for customer in self.customers.values()
            if customer.total_revenue >= min_total_revenue
        ]

        # 2. Sort them by revenue, then by ID
        eligible.sort(key=lambda c: (c.total_revenue, c.id))

        # 3. Return the first k IDs
        return set(c.id for c in eligible[:k])



### Part 2
!pip install sortedcontainers
from sortedcontainers import SortedList

class Customer:
    def __init__(self, id: int, revenue: int):
        self.id = id
        self.revenue = revenue
        self.total_revenue = revenue
        self.referrals = []

    def __lt__(self, other):
        if self.total_revenue != other.total_revenue:
            return self.total_revenue < other.total_revenue
        return self.id < other.id

    def __eq__(self, other):
        return self.id == other.id

    def __hash__(self):
        return hash(self.id)

class RevenueSystem:
    def __init__(self):
        self.customers = {}
        self.sorted_customers = SortedList()
        self.next_id = 0

    def insert(self, revenue: int, referrer_id: int = None) -> int:
        """
        Add a customer and update the sorted list.
        Time Complexity: O(log n)
        """
        customer_id = self.next_id
        self.next_id += 1

        customer = Customer(customer_id, revenue)
        self.customers[customer_id] = customer
        self.sorted_customers.add(customer)

        if referrer_id is not None:
            referrer = self.customers[referrer_id]

            # We must remove the referrer before updating their value
            # so the SortedList doesn't get confused
            self.sorted_customers.remove(referrer)

            # Update revenue
            referrer.total_revenue += revenue
            referrer.referrals.append(customer_id)

            # Put them back in the list in the new correct position
            self.sorted_customers.add(referrer)

        return customer_id
## Part 2 Heavy READS
    def get_lowest_k_by_total_revenue(self, k: int, min_total_revenue: int) -> set:
        """
        Get results quickly from the pre-sorted list.
        Time Complexity: O(n) worst case, O(k) best case
        """
        result = set()

        # Just walk through the already sorted list
        for customer in self.sorted_customers:
            if customer.total_revenue < min_total_revenue:
                continue

            result.add(customer.id)
            if len(result) == k:
                break

        return result


In [ ]:
def get_lowest_k_by_total_revenue(self, k: int, min_total_revenue: int) -> set:
    # Create a fake customer to find the search position
    dummy = Customer(-1, min_total_revenue)

    # Find the index where revenue >= min_total_revenue
    start_idx = self.sorted_customers.bisect_left(dummy)

    # Grab the next k items starting from that index
    result = set()
    for i in range(start_idx, len(self.sorted_customers)):
        if len(result) == k:
            break
        result.add(self.sorted_customers[i].id)

    return result

In [ ]:
class Customer:
    def __init__(self, id: int, revenue: int):
        self.id = id
        self.revenue = revenue
        self.total_revenue = revenue
        self.referrals = []

class RevenueSystem:
    def __init__(self):
        self.customers = {}       # Map ID -> Customer object
        self.next_id = 0
        self.sorted_cache = None  # Stores the sorted list for repeated queries

    def insert(self, revenue: int, referrer_id: int = None) -> int:
        """
        Add a new customer.
        Time Complexity: O(1)
        """
        customer_id = self.next_id
        self.next_id += 1

        customer = Customer(customer_id, revenue)
        self.customers[customer_id] = customer

        if referrer_id is not None:
            # Update the person who referred them
            if referrer_id in self.customers:
                referrer = self.customers[referrer_id]
                referrer.total_revenue += revenue
                referrer.referrals.append(customer_id)
            else:
                # Optional: Handle case where referrer doesn't exist
                pass

        # We changed data, so the old sorted list is now wrong.
        # Set it to None so the next query knows to re-sort.
        self.sorted_cache = None

        return customer_id

    def get_lowest_k_by_total_revenue(self, k: int, min_total_revenue: int) -> set:
        """
        Get lowest k revenue customers.
        Time Complexity:
          - O(N log N) if insert() was called recently.
          - O(K) if called multiple times in a row (uses cache).
        """
        # 1. Check if we need to re-sort
        if self.sorted_cache is None:
            # Sort all customers: primarily by total_revenue, secondarily by ID
            self.sorted_cache = sorted(
                self.customers.values(),
                key=lambda c: (c.total_revenue, c.id)
            )

        # 2. Iterate through the cached sorted list
        result = set()
        for customer in self.sorted_cache:
            if customer.total_revenue >= min_total_revenue:
                result.add(customer.id)
                if len(result) == k:
                    break

        return result

In [ ]:
### POST: https://www.1point3acres.com/interview/problems/post/7100018